In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np
import re

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

def find_timing_file(run_folder_path: str, sim_type: str) -> Optional[str]:
    """Finds the ...trace_matched_timing.csv file for a given run."""
    sim_output_dir = os.path.join(run_folder_path, sim_type.lower())
    if not os.path.isdir(sim_output_dir):
        return None
    for f in os.listdir(sim_output_dir):
        if 'trace_matched_timing.csv' in f:
            return os.path.join(sim_output_dir, f)
    return None


In [2]:
import re
import pandas as pd
from plotly.subplots import make_subplots

# --- Configuration ---
base_comparison_folders = [
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/multiple_collectives/'
]
comparison_plot_metric = 'max'  # Can be 'avg' or 'max'

# --- Helper Functions ---
cc_modes = {0: "PFC", 1: "DCQCN", 3: "HPCC", 7: "TIMELY", 8: "DCTCP", 10: "HPCC-PINT"}
topo_idx_regex = re.compile(r'(?:ns3|G2)_FoldedClos_16_v(\d+)_Random(?:\.json)?$')

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        h, m, s = map(float, time_str.split(':'))
        return h * 3600 + m * 60 + s
    except (ValueError, IndexError):
        return 0.0

def process_simulation(folder: str, run_folder: str, sim_type: str, summary_params: dict, workload_name: str, topo_index: int, run_name: str) -> Optional[dict]:
    """Processes a single simulation run, extracts timing data, and returns a result dictionary."""
    timing_file = find_timing_file(run_folder, sim_type)
    if not timing_file:
        return None

    try:
        df = pd.read_csv(timing_file)
        if sim_type == 'ns3' and 'node_name' in df.columns:
            df = df[df['node_name'] != 'dummy_node'].copy()

        time_col = 'callback_tick'
        if time_col not in df.columns:
            return None

        df = df[df[time_col] > 100].copy()
        elapsed_times = df[time_col].dropna()
        if elapsed_times.empty:
            return None

        return {
            'workload': workload_name,
            'npu_count': summary_params.get('npus count', 'N/A'),
            'topo_index': topo_index,
            'sim_type': sim_type.upper(),
            'run_name': run_name,
            'avg_time': elapsed_times.mean(),
            'max_time': elapsed_times.max(),
            'min_time': elapsed_times.min(),
            'std_dev': elapsed_times.std(),
            'execution_time': parse_runtime(summary_params.get('total runtime', '0:0:0.0')),
            'folder': folder,
            'run_folder': run_folder
        }
    except Exception as e:
        print(f"Error processing {sim_type} in {folder}: {e}")
        return None

# --- Data Collection Logic ---
all_run_folders = []
for base_folder in base_comparison_folders:
    for workload_folder in os.listdir(base_folder):
        workload_path = os.path.join(base_folder, workload_folder)
        if os.path.isdir(workload_path):
            all_run_folders.extend([(workload_path, os.path.join(workload_path, d)) for d in os.listdir(workload_path) if os.path.isdir(os.path.join(workload_path, d))])

comparison_results = []
for folder, run_folder in sorted(all_run_folders):
    run_summary_path = os.path.join(run_folder, 'run_summary.txt')
    if not os.path.exists(run_summary_path):
        continue

    summary_params = parse_config(run_summary_path)
    workload_name = os.path.basename(os.path.dirname(run_folder))
    
    # Process all potential simulation types in the folder
    sim_dirs = [d for d in os.listdir(run_folder) if os.path.isdir(os.path.join(run_folder, d)) and d in ['analytical_unaware', 'g2', 'ns3']]

    for sim_type in sim_dirs:
        topo_index = -1
        run_name = f"{sim_type.replace('_', ' ').title()}"

        if sim_type != 'analytical_unaware':
            topo_key = 'g2 topology file override' if sim_type == 'g2' else 'ns3 topology file override'
            topo_file = summary_params.get(topo_key, '')
            if 'all_paths' in topo_file: continue
            
            match = topo_idx_regex.search(topo_file)
            if not match: continue
            topo_index = int(match.group(1))

            if sim_type == 'ns3':
                ns3_config_file = find_config_file(run_folder)
                if ns3_config_file:
                    ns3_params = parse_config(ns3_config_file)
                    run_name = (
                        f"NS3 (cc:{cc_modes.get(int(ns3_params.get('cc_mode', -1)), 'N/A')}, "
                        f"win:{ns3_params.get('has_win', 'N/A')}, adapt:{ns3_params.get('var_win', 'N/A')}, "
                        f"buf:{ns3_params.get('buffer_size', 'N/A')}, size:{ns3_params.get('packet_payload_size', 'N/A')})"
                    )
            else: # G2
                run_name = "G2"

        result = process_simulation(folder, run_folder, sim_type, summary_params, workload_name, topo_index, run_name)
        if result:
            comparison_results.append(result)

# --- Plotting Logic ---


In [5]:
# --- Analysis and Plotting ---
if comparison_results:
    comp_df = pd.DataFrame(comparison_results)

    def analyze_and_plot_divergence(metric_col, metric_name):
        print(f"--- Analyzing Divergence for: {metric_name} ---")
        
        divergence_data = []
        for (wl, ti), group in comp_df.groupby(['workload', 'topo_index']):
            g2_runs = group[group['sim_type'] == 'G2']
            ns3_runs = group[group['sim_type'] == 'NS3']

            if g2_runs.empty or ns3_runs.empty:
                continue

            g2_time = g2_runs.iloc[0][metric_col]
            best_ns3_run = ns3_runs.loc[ns3_runs[metric_col].idxmin()]
            best_ns3_time = best_ns3_run[metric_col]

            if g2_time < 100 or best_ns3_time < 100:
                continue

            divergence = abs((g2_time - best_ns3_time)/g2_time)
            
            # Extract info from workload name
            match = re.match(r'([a-zA-Z_]+)_size_(\d+)_group_(\d+)', wl)
            comm_type, comm_size, group_id = "N/A", "N/A", "N/A"
            if match:
                comm_type = match.group(1).replace('_', ' ').title()
                comm_size = int(match.group(2))
                group_id = int(match.group(3))

            divergence_data.append({
                'Workload': wl,
                'Topo_index': ti,
                'Divergence': divergence,
                'G2 Time': g2_time,
                'Best NS3 Time': best_ns3_time,
                'Best NS3 Run': best_ns3_run['run_name'],
                'comm_type': comm_type,
                'comm_size': comm_size,
                'group_id': group_id,
            })

        if not divergence_data:
            print(f"No divergent workloads found for {metric_name}.\n")
            return

        div_df = pd.DataFrame(divergence_data).sort_values(by='Divergence', ascending=False)
        top_10_workloads = div_df.head(10)

        print(f"\n--- Top 10 Most Divergent Workloads ({metric_name}) ---")
        with pd.option_context('display.float_format', '{:,.2f}'.format):
            display(top_10_workloads[['Workload', 'comm_type', 'comm_size', 'group_id', 'G2 Time', 'Best NS3 Time', 'Divergence', 'Topo_index']])

        print(f"\n--- Generating Plots for Top 10 Divergent Workloads ({metric_name}) ---")
        for _, row in top_10_workloads.iterrows():
            wl = row['Workload']
            topo_index = row['Topo_index']
            
            workload_df = comp_df[(comp_df['workload'] == wl) & (comp_df['topo_index'] == topo_index)]
        
            group_df = workload_df

            g2_run = group_df[group_df['sim_type'] == 'G2'].sort_values(by=metric_col).drop_duplicates(subset=['run_name'], keep='first')
            ns3_runs_for_plot = group_df[group_df['sim_type'] == 'NS3'].sort_values(by=metric_col).drop_duplicates(subset=['run_name'], keep='first')
            best_ns3_run_for_plot = ns3_runs_for_plot.iloc[0]

            fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Performance ({metric_name})", "Per-NPU Time Correlation"))

            # Bar plot
            fig.add_trace(go.Bar(x=ns3_runs_for_plot['run_name'], y=ns3_runs_for_plot[metric_col], name='NS3 Runs'), row=1, col=1)
            fig.add_hline(y=
            g2_run[metric_col].values[0], line_dash="dot", annotation_text=f"G2 Time", row=1, col=1)

            # Scatter plot
            g2_timing_file = find_timing_file(g2_run['run_folder'].values[0], 'G2')
            ns3_timing_file = find_timing_file(best_ns3_run_for_plot['run_folder'], 'NS3')
            if g2_timing_file and ns3_timing_file:
                try:
                    df_g2 = pd.read_csv(g2_timing_file)[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'g2_time'})
                    df_ns3 = pd.read_csv(ns3_timing_file)[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'ns3_time'})
                    merged_df = pd.merge(df_ns3, df_g2, on='sys_id')
                    merged_df = merged_df[(merged_df['ns3_time'] >= 100) & (merged_df['g2_time'] >= 100)]
                    
                    fig.add_trace(go.Scatter(x=merged_df['ns3_time'], y=merged_df['g2_time'], mode='markers', name='NPU Times'), row=1, col=2)
                    min_val = min(merged_df['ns3_time'].min(), merged_df['g2_time'].min())
                    max_val = max(merged_df['ns3_time'].max(), merged_df['g2_time'].max())
                    fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode='lines', name='y=x'), row=1, col=2)
                except Exception as e:
                    print(f"Could not create scatter plot for {wl}: {e}")
            
            plot_title = (f'<b>{wl}</b><br>'
                          f'Metric: {metric_name}<br>'
                          f'Comm: {row["comm_type"]}, Size: {row["comm_size"]}, Group: {row["group_id"]}')

            fig.update_layout(title_text=plot_title, height=700, margin=dict(t=140))
            fig.show()
        print("\n" + "="*80 + "\n")
        return div_df


    # --- Run analysis for each metric ---
    div_df_avg = analyze_and_plot_divergence('avg_time', 'Average Time')
    div_df_max = analyze_and_plot_divergence('max_time', 'Maximum Time')
    div_df_min = analyze_and_plot_divergence('min_time', 'Minimum Time')

else:
    print("No comparison results to process.")


--- Analyzing Divergence for: Average Time ---

--- Top 10 Most Divergent Workloads (Average Time) ---


,Workload,comm_type,comm_size,group_id,G2 Time,Best NS3 Time,Divergence,Topo_index
15,all_gather_size_33554432_group_1,All Gather,33554432,1,"4,203,742.00","5,119,980.25",0.22,2
28,all_reduce_size_134217728_group_1,All Reduce,134217728,1,"8,497,618.00","10,275,328.75",0.21,2
69,reduce_scatter_size_134217728_group_1,Reduce Scatter,134217728,1,"4,293,886.00","5,153,017.88",0.20,2
56,all_to_all_size_33554432_group_1,All To All,33554432,1,"2,495,957.00","2,056,621.88",0.18,4
46,all_to_all_size_1048576_group_1,All To All,1048576,1,"78,017.00","64,496.25",0.17,4
38,all_reduce_size_33554432_group_1,All Reduce,33554432,1,"2,482,810.25","2,058,316.38",0.17,4
25,all_reduce_size_1048576_group_1,All Reduce,1048576,1,"77,603.75","64,566.75",0.17,4
17,all_gather_size_33554432_group_1,All Gather,33554432,1,"4,203,726.25","4,737,863.12",0.13,4
71,reduce_scatter_size_134217728_group_1,Reduce Scatter,134217728,1,"4,248,798.25","4,782,827.50",0.13,4
6,all_gather_size_1048576_group_1,All Gather,1048576,1,"65,722.00","73,208.88",0.11,3



--- Generating Plots for Top 10 Divergent Workloads (Average Time) ---




--- Analyzing Divergence for: Maximum Time ---

--- Top 10 Most Divergent Workloads (Maximum Time) ---


,Workload,comm_type,comm_size,group_id,G2 Time,Best NS3 Time,Divergence,Topo_index
28,all_reduce_size_134217728_group_1,All Reduce,134217728,1,8497618,10631639,0.25,2
15,all_gather_size_33554432_group_1,All Gather,33554432,1,4203742,5184668,0.23,2
69,reduce_scatter_size_134217728_group_1,Reduce Scatter,134217728,1,4293886,5216215,0.21,2
7,all_gather_size_1048576_group_1,All Gather,1048576,1,164232,139570,0.15,4
79,reduce_scatter_size_33554432_group_1,Reduce Scatter,33554432,1,1324948,1129926,0.15,4
47,all_to_all_size_134217728_group_0,All To All,134217728,0,8407474,9610287,0.14,1
64,reduce_scatter_size_1048576_group_1,Reduce Scatter,1048576,1,41431,35512,0.14,4
16,all_gather_size_33554432_group_1,All Gather,33554432,1,2101891,2369953,0.13,3
6,all_gather_size_1048576_group_1,All Gather,1048576,1,65722,74075,0.13,3
56,all_to_all_size_33554432_group_1,All To All,33554432,1,2572580,2262729,0.12,4



--- Generating Plots for Top 10 Divergent Workloads (Maximum Time) ---




--- Analyzing Divergence for: Minimum Time ---

--- Top 10 Most Divergent Workloads (Minimum Time) ---


,Workload,comm_type,comm_size,group_id,G2 Time,Best NS3 Time,Divergence,Topo_index
56,all_to_all_size_33554432_group_1,All To All,33554432,1,2386492,1696926,0.29,4
46,all_to_all_size_1048576_group_1,All To All,1048576,1,74602,53516,0.28,4
12,all_gather_size_33554432_group_0,All Gather,33554432,0,3503125,4491181,0.28,3
38,all_reduce_size_33554432_group_1,All Reduce,33554432,1,2370528,1709506,0.28,4
25,all_reduce_size_1048576_group_1,All Reduce,1048576,1,74101,53947,0.27,4
13,all_gather_size_33554432_group_0,All Gather,33554432,0,3503125,4438992,0.27,4
2,all_gather_size_1048576_group_0,All Gather,1048576,0,109510,138548,0.27,3
67,reduce_scatter_size_134217728_group_0,Reduce Scatter,134217728,0,3593269,4512121,0.26,3
74,reduce_scatter_size_33554432_group_0,Reduce Scatter,33554432,0,898346,1126923,0.25,3
59,reduce_scatter_size_1048576_group_0,Reduce Scatter,1048576,0,28112,35235,0.25,3



--- Generating Plots for Top 10 Divergent Workloads (Minimum Time) ---


In [4]:
pd.DataFrame(comparison_results).head()

,workload,npu_count,topo_index,sim_type,run_name,avg_time,max_time,min_time,std_dev,execution_time,folder,run_folder
0,all_gather_size_1048576_group_0,16,-1,ANALYTICAL_UNAWARE,Analytical Unaware,1.396984e+09,1396983901,1396983901,0.000000,1.858392,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
1,all_gather_size_1048576_group_0,16,1,G2,G2,1.314070e+05,131407,131407,0.000000,1.040386,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
2,all_gather_size_1048576_group_0,16,2,G2,G2,1.204485e+05,131396,109501,11703.369783,1.094770,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
3,all_gather_size_1048576_group_0,16,3,G2,G2,1.368680e+05,153299,109510,19404.219733,1.150742,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
4,all_gather_size_1048576_group_0,16,4,G2,G2,1.368680e+05,153299,109510,19404.219733,1.079302,/app/astra-sim/upc/output/comparison_run/Folde...,/app/astra-sim/upc/output/comparison_run/Folde...
